# Experiment 5: Backward Chaining Engine with Askable Facts
**Aim:** Implement goal-driven backward chaining for car-fault diagnosis using askable facts.

## Knowledge Base

In [2]:
# Rules: fault -> conditions ("~" = negation)
RULES = {
    "dead_battery":      ["~engine_cranks", "headlights_dim"],
    "starter_motor":     ["~engine_cranks", "clicking_sound"],
    "out_of_fuel":       ["engine_cranks", "fuel_gauge_empty"],
    "spark_plug_fault":  ["engine_cranks", "rough_idle", "misfire"],
    "overheating":       ["temp_high", "coolant_low"],
}

# Askable facts
QUESTIONS = {
    "engine_cranks":    "Does the engine crank when you turn the key?",
    "headlights_dim":   "Are the headlights dim?",
    "fuel_gauge_empty": "Is the fuel gauge on empty?",
    "rough_idle":       "Does the engine idle roughly?",
    "misfire":          "Is the engine misfiring?",
    "temp_high":        "Is the temperature gauge high?",
    "coolant_low":      "Is the coolant level low?",
    "clicking_sound":   "Do you hear a clicking sound on starting?",
}

## Engine

In [3]:
SCRIPT = []   # preset answers (optional); empty -> uses input()

def ask(fact, rule, head):
    while True:
        if SCRIPT:
            a = SCRIPT.pop(0); print(f"? {QUESTIONS[fact]} (y/n/why) -> {a}")
        else:
            a = input(f"? {QUESTIONS[fact]} (y/n/why): ")
        a = a.strip().lower()
        if a == "why":
            print(f"   WHY: testing rule  IF {' AND '.join(rule)} THEN {head}")
        elif a in ("y", "n"):
            return a == "y"

def prove(goal, known, asked, rule=None, head=None):
    neg = goal.startswith("~")
    fact = goal.lstrip("~")
    if fact in QUESTIONS:
        if fact not in known:
            known[fact] = ask(fact, rule, head); asked.append(fact)
        return known[fact] != neg
    return all(prove(c, known, asked, RULES[fact], fact) for c in RULES[fact])

def diagnose(hypotheses):
    known, asked = {}, []
    for h in hypotheses:
        if all(prove(c, known, asked, RULES[h], h) for c in RULES[h]):
            print(f"\nDiagnosis: {h}"); break
    else:
        print("\nNo fault identified")
    print("Facts asked:", asked)

## Exercise 1: Starter motor fault

In [4]:
SCRIPT = ["n", "n", "y"]   # not cranking, lights ok, clicking
diagnose(list(RULES))

? Does the engine crank when you turn the key? (y/n/why) -> n
? Are the headlights dim? (y/n/why) -> n
? Do you hear a clicking sound on starting? (y/n/why) -> y

Diagnosis: starter_motor
Facts asked: ['engine_cranks', 'headlights_dim', 'clicking_sound']


**Observation:** Battery rule fails at `headlights_dim`; starter rule reuses `engine_cranks` and asks only `clicking_sound`.

## Exercise 2: Change hypothesis order

In [5]:
SCRIPT = ["n", "y"]   # not cranking, clicking
diagnose(["starter_motor", "dead_battery", "out_of_fuel", "spark_plug_fault", "overheating"])

? Does the engine crank when you turn the key? (y/n/why) -> n
? Do you hear a clicking sound on starting? (y/n/why) -> y

Diagnosis: starter_motor
Facts asked: ['engine_cranks', 'clicking_sound']


**Observation:** Starter is tried first, so `headlights_dim` is never asked; questions depend on hypothesis order.

## Exercise 3: WHY option

In [6]:
SCRIPT = ["n", "why", "n", "y"]   # ask WHY at headlights_dim
diagnose(list(RULES))

? Does the engine crank when you turn the key? (y/n/why) -> n
? Are the headlights dim? (y/n/why) -> why
   WHY: testing rule  IF ~engine_cranks AND headlights_dim THEN dead_battery
? Are the headlights dim? (y/n/why) -> n
? Do you hear a clicking sound on starting? (y/n/why) -> y

Diagnosis: starter_motor
Facts asked: ['engine_cranks', 'headlights_dim', 'clicking_sound']


**Observation:** WHY prints the rule under test that caused the question.

## Interactive Run

In [7]:
SCRIPT = []
diagnose(list(RULES))


Diagnosis: starter_motor
Facts asked: ['engine_cranks', 'headlights_dim', 'clicking_sound']
